In [0]:
%sql
-- Databricks notebook source
-- Top 5 operadoras por beneficiários ativos
-- Pega a Silver, soma os beneficiários ativos por operadora e monta o ranking. Fica uma tabela pronta, sem precisar recalcular toda vez que alguém for consultar.

In [0]:
%sql

USE CATALOG bmg_saude_desafio;
USE SCHEMA gold;

In [0]:
%sql
-- cria (ou substitui) a tabela já com o ranking calculado
CREATE OR REPLACE TABLE gold.top_operadoras_beneficiarios_ativos
USING DELTA
COMMENT 'Ranking de operadoras por beneficiarios ativos'
AS
SELECT
  cd_operadora,
  nm_razao_social,
  SUM(qt_beneficiario_ativo) AS total_beneficiarios_ativos,
  RANK() OVER (ORDER BY SUM(qt_beneficiario_ativo) DESC) AS ranking
FROM silver.beneficiarios_ans
GROUP BY cd_operadora, nm_razao_social
ORDER BY total_beneficiarios_ativos DESC;

In [0]:
%sql
-- Select top 5
SELECT
    ranking,
    cd_operadora,
    nm_razao_social,
    REPLACE(FORMAT_NUMBER(total_beneficiarios_ativos, 0), ',', '.') AS total_beneficiarios_ativos
FROM gold.top_operadoras_beneficiarios_ativos
WHERE ranking <= 5
ORDER BY ranking;

In [0]:
%sql
-- MAGIC %md
-- MAGIC Resultado esperado (já validado com os dados reais, competência
-- MAGIC 2025-08, Tocantins):
-- MAGIC
-- MAGIC | # | Operadora | Beneficiários ativos |
-- MAGIC |---|---|---|
-- MAGIC | 1 | PREVIDENT ASSISTÊNCIA ODONTOLÓGICA S.A | 67.430 |
-- MAGIC | 2 | ODONTOPREV S/A | 52.832 |
-- MAGIC | 3 | UNIMED PALMAS COOPERATIVA DE TRABALHO MÉDICO | 29.354 |
-- MAGIC | 4 | BRADESCO SAÚDE S.A. | 16.280 |
-- MAGIC | 5 | COOPERATIVA DE TRABALHO MEDICO DE ARAGUAÍNA - UNIMED ARAGUAÍNA | 14.353 |